In [1]:
# ============================================
# Gloo API Discovery Notebook
# ============================================

# STEP 1:
# Install dependencies if needed:
#
# !pip install requests python-dotenv
#
# ============================================

import os
import json
import requests
from dotenv import load_dotenv

# ============================================
# Load .env
# ============================================

load_dotenv()

CLIENT_ID = os.getenv("CLIENT_ID")
CLIENT_SECRET = os.getenv("CLIENT_SECRET")
PUBLISHER_ID = os.getenv("PUBLISHER_ID")

print("CLIENT_ID:", CLIENT_ID)
print("CLIENT_SECRET EXISTS:", bool(CLIENT_SECRET))
print("PUBLISHER_ID:", PUBLISHER_ID)

# ============================================
# URLs
# ============================================

TOKEN_URL = "https://platform.ai.gloo.com/oauth2/token"

# Candidate endpoints to discover
DISCOVERY_ENDPOINTS = [
    "https://platform.ai.gloo.com/ai/data/v1/collections",
    "https://platform.ai.gloo.com/ai/data/v1/tenants",
    "https://platform.ai.gloo.com/ingestion/v2/files",
    "https://platform.ai.gloo.com/ai/data/v1/search"
]

# ============================================
# STEP 2:
# Get OAuth Token
# ============================================

print("\nGetting access token...\n")

response = requests.post(
    TOKEN_URL,
    auth=(CLIENT_ID, CLIENT_SECRET),
    headers={
        "Content-Type": "application/x-www-form-urlencoded"
    },
    data={
        "grant_type": "client_credentials"
    },
    timeout=30
)

print("TOKEN STATUS:", response.status_code)

if response.status_code != 200:
    print(response.text)
    raise Exception("Failed to get token")

token_data = response.json()

ACCESS_TOKEN = token_data["access_token"]

print("\nACCESS TOKEN ACQUIRED\n")

# ============================================
# STEP 3:
# Test Discovery Endpoints
# ============================================

headers = {
    "Authorization": f"Bearer {ACCESS_TOKEN}"
}

for endpoint in DISCOVERY_ENDPOINTS:

    print("\n" + "=" * 80)
    print("TESTING:", endpoint)
    print("=" * 80)

    try:

        if "search" in endpoint:

            # Search endpoint requires payload
            resp = requests.post(
                endpoint,
                headers={
                    **headers,
                    "Content-Type": "application/json"
                },
                json={
                    "query": "test",
                    "collection": "test",
                    "tenant": "test",
                    "limit": 1
                },
                timeout=30
            )

        else:

            resp = requests.get(
                endpoint,
                headers=headers,
                timeout=30
            )

        print("STATUS:", resp.status_code)

        try:
            data = resp.json()

            print("\nRESPONSE JSON:\n")
            print(json.dumps(data, indent=2))

        except:
            print("\nRAW RESPONSE:\n")
            print(resp.text[:3000])

    except Exception as e:

        print("ERROR:", str(e))

# ============================================
# STEP 4:
# Optional Upload Test Metadata
# ============================================

print("\n" + "=" * 80)
print("OPTIONAL: TEST FILE UPLOAD RESPONSE")
print("=" * 80)

# Create temporary file
temp_filename = "gloo_test.txt"

with open(temp_filename, "w", encoding="utf-8") as f:
    f.write("""
This is a test upload.

Potential metadata discovery:
- tenant
- collection
- indexing response
""")

upload_url = "https://platform.ai.gloo.com/ingestion/v2/files"

with open(temp_filename, "rb") as f:

    files = {
        "files": (
            temp_filename,
            f,
            "text/plain"
        )
    }

    data = {
        "publisher_id": PUBLISHER_ID
    }

    upload_resp = requests.post(
        upload_url,
        headers={
            "Authorization": f"Bearer {ACCESS_TOKEN}"
        },
        files=files,
        data=data,
        timeout=120
    )

print("UPLOAD STATUS:", upload_resp.status_code)

try:

    upload_json = upload_resp.json()

    print("\nUPLOAD RESPONSE JSON:\n")
    print(json.dumps(upload_json, indent=2))

except:

    print("\nRAW RESPONSE:\n")
    print(upload_resp.text)

# Cleanup
os.remove(temp_filename)

print("\nDONE")

CLIENT_ID: 7k0k6rde0oquas7mvhjvjqf218
CLIENT_SECRET EXISTS: True
PUBLISHER_ID: c5162eff-7797-47eb-915a-7765f74718df

Getting access token...

TOKEN STATUS: 200

ACCESS TOKEN ACQUIRED


TESTING: https://platform.ai.gloo.com/ai/data/v1/collections
STATUS: 404

RESPONSE JSON:

{
  "detail": "Not Found"
}

TESTING: https://platform.ai.gloo.com/ai/data/v1/tenants
STATUS: 404

RESPONSE JSON:

{
  "detail": "Not Found"
}

TESTING: https://platform.ai.gloo.com/ingestion/v2/files
STATUS: 405

RESPONSE JSON:

{
  "detail": "Method Not Allowed"
}

TESTING: https://platform.ai.gloo.com/ai/data/v1/search
STATUS: 403

RESPONSE JSON:

{
  "error": "Forbidden - insufficient permissions",
  "code": "forbidden"
}

OPTIONAL: TEST FILE UPLOAD RESPONSE
UPLOAD STATUS: 200

UPLOAD RESPONSE JSON:

{
  "success": true,
  "message": "File processing started in background.",
  "ingesting": [
    "a2f1b3b8-fd26-4718-a8a0-b82314cb1d40"
  ],
  "duplicates": []
}

DONE


In [2]:
import requests
import json

SEARCH_URL = "https://platform.ai.gloo.com/ai/data/v1/search"

payload = {
    "query": "What is 2414 World?",
    "limit": 5
}

response = requests.post(
    SEARCH_URL,
    headers={
        "Authorization": f"Bearer {ACCESS_TOKEN}",
        "Content-Type": "application/json"
    },
    json=payload
)

print("STATUS:", response.status_code)

try:
    print(json.dumps(response.json(), indent=2))
except:
    print(response.text)

STATUS: 403
{
  "error": "Forbidden - insufficient permissions",
  "code": "forbidden"
}
